# Lesson 2.3 - Flux Bootstrap And Terraform Outputs

**Level: Intermediate**

## Learning Objectives
- Install Flux extension and Flux configuration from Terraform.
- Point Flux at the bootstrap path in this repo.
- Add outputs that simplify post-apply operations.

**Project file(s) produced:** `terraform/foundation/flux.tf`, `terraform/foundation/outputs.tf`, `terraform/foundation/terraform.tfvars.example`

## Concepts
Flux is installed as an AKS extension, then configured to reconcile `./bootstrap`.
Terraform outputs provide quick commands and IDs needed for operations.

In [ ]:
%%writefile terraform/foundation/flux.tf
resource "azurerm_kubernetes_cluster_extension" "flux" {
  name           = local.names.flux_extension
  cluster_id     = azurerm_kubernetes_cluster.this.id
  extension_type = "microsoft.flux"

  timeouts {
    create = "60m"
    delete = "60m"
  }
}

resource "azurerm_kubernetes_flux_configuration" "bootstrap" {
  name       = local.names.flux_config
  cluster_id = azurerm_kubernetes_cluster.this.id
  namespace  = "flux-system"

  git_repository {
    url                    = var.flux_git_repository_url
    reference_type         = "branch"
    reference_value        = var.flux_git_branch
    ssh_private_key_base64 = var.flux_git_ssh_private_key_base64
  }

  kustomizations {
    name = "bootstrap"
    path = var.flux_bootstrap_path
  }

  depends_on = [azurerm_kubernetes_cluster_extension.flux]
}

In [ ]:
%%writefile terraform/foundation/outputs.tf
output "resource_group_name" { value = local.resource_group_name }
output "aks_cluster_name" { value = azurerm_kubernetes_cluster.this.name }
output "aks_get_credentials_command" {
  value = "az aks get-credentials --resource-group ${local.resource_group_name} --name ${azurerm_kubernetes_cluster.this.name} --overwrite-existing"
}
output "key_vault_name" { value = azurerm_key_vault.this.name }
output "key_vault_uri" { value = azurerm_key_vault.this.vault_uri }
output "flux_configuration_id" { value = azurerm_kubernetes_flux_configuration.bootstrap.id }

In [ ]:
%%writefile terraform/foundation/terraform.tfvars.example
resource_group_name = null
flux_git_repository_url = "git@github.com:your-org/your-repo.git"
flux_git_branch = "main"
flux_bootstrap_path = "./bootstrap"
gpu_node_pool_enabled = true

## Exercise
Fill in a local `terraform.tfvars` based on your Azure subscription constraints (region, RG, GPU enabled/disabled).

In [ ]:
# TODO: Copy terraform.tfvars.example to terraform.tfvars
# TODO: Set real flux_git_repository_url and branch
# TODO: Export TF_VAR_flux_git_ssh_private_key_base64 before apply
print('Then run: terraform -chdir=terraform/foundation init && terraform -chdir=terraform/foundation plan')

---
**Next:** [Lesson 3.1 - Bootstrap Namespaces And Helm Source](../phase-3-flux-argocd-bootstrap/lesson-3.1-bootstrap-namespaces-and-helm-source.ipynb)